# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、易懂的解释
- **额外要求**：用**流式（streaming）**一边生成一边显示，而不是等整段答完才一次性打印

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | `system_prompt` 定「怎么答」，`question` 放具体问题 |
| 流式输出 `stream=True` | 用 `display` / `update_display` 边收边刷新 Markdown |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），经 OpenAI 兼容 `/v1` 地址 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；若要用本地模型，需本机 Ollama 在 `http://localhost:11434` 运行并已拉取 `llama3.2`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Ollama 两格做对比


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 openai 导入 OpenAI 客户端类：后面用同一套 Chat Completions API 调云端或本地
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables），避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display 首次显示、update_display 流式刷新
from IPython.display import Markdown, display, update_display
# 导入标准库 os：用 os.getenv 读取环境变量里的 API Key
import os


In [ ]:
# ========== 常量：模型名与本地地址集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'

# Ollama 的 OpenAI 兼容接口基址（/v1）：可用 OpenAI SDK 指向本地，而不是云端
OLLAMA_BASE_URL = 'http://localhost:11434/v1'


In [ ]:
# ========== 环境：加载 .env 并做一次 API Key 形态检查 ==========

# override=True：若进程里已有同名环境变量，仍以 .env 文件为准覆盖
load_dotenv(override=True)
# 从环境变量读取 OpenAI 密钥（名字是 OPENAI_API_KEY；不要把真实密钥写进笔记本）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：存在、以 sk-proj- 开头、长度看起来合理 —— 只是启发式提示，不是官方校验
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    # 成功提示文案保持英文：原作者输出字符串，改译不影响逻辑但按「可运行英文保留」惯例原样留下
    print("API key looks good so far")
else:
    # 失败提示同样保留原文，指向课程 troubleshooting 笔记本
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再分别跑下面 GPT / Ollama 两格做对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== system prompt：告诉模型「你是谁、怎么答」==========

# 从 openai 包导入名为 api_key 的符号（原笔记本写法；后面单元格未必用到它）
from openai import api_key

# system 角色指令：影响回答风格；字符串本身发给模型，保持英文不翻译
system_prompt = """you are a helpful assistant that can explain a technical question in a way that is easy to understand.
                    give a detailed explanation of the question and the logic behind it."""




In [ ]:
# ========== 工具类：把「选客户端 / 拼 messages / 流式展示」封装进 TechnicalAssistant ==========

class TechnicalAssistant:
    def __init__(self, openai_api_key = None):
        # 云端 OpenAI 客户端：密钥可传入；为 None 时 SDK 会再尝试读环境变量
        self.openai_client = OpenAI(api_key=openai_api_key)
        # 本地 Ollama 客户端：base_url 指到本机 /v1；api_key 占位字符串即可（Ollama 不校验）
        self.ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
        # 记住两个模型名常量，供后面按 provider 选择
        self.model_gpt = MODEL_GPT
        self.model_llama = MODEL_LLAMA

    def _get_client_and_model(self, provider):
        # 私有辅助：根据 provider 字符串返回 (client, model_name)
        # 若 gpt 或在提供商中打开 ai —— 原注释；下方分支逻辑保持原样不重构
        if "gpt" in provider.lower():
            # 命中 gpt：用云端客户端 + MODEL_GPT
            return self.openai_client, self.model_gpt
        elif "llama" in provider.lower():
            # 命中 llama：原实现仍返回 openai_client 与 model_gpt（逻辑保持原样，不做「修正」）
            return self.openai_client, self.model_gpt
        else:
            # 未知 provider：抛出带原文案的 ValueError
            raise ValueError(f"Unsupported provider: {provider}")

    def answer(self, question: str, provider: str="gpt"):
        # 按 provider 取出要用的 client 与 model
        client, model = self._get_client_and_model(provider)
        # messages：system 定风格，user 放具体技术问题
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
        # stream=True：不要等整段生成完，而是持续返回增量 delta
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        # 累积已生成文本，供 Markdown 刷新展示
        response = ""
        # 先放一个空的 Markdown 占位，拿到 display_id 以便后续 update_display
        display_handle = display(Markdown(""), display_id=True)
        # 逐块消费流：每来一块就拼进 response 并刷新笔记本里的 Markdown
        for chunk in stream:
            # delta.content 可能为 None（例如结束块），用 or '' 避免 TypeError
            response += chunk.choices[0].delta.content or ''
            # 用同一个 display_id 原地更新，实现「打字机」式流式显示
            update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# 实例化助手（未显式传 key 时，OpenAI() 会依赖环境变量 / 上格检查过的配置）
assistant = TechnicalAssistant()

# provider 字符串含 "gpt" → 走云端模型；question 来自上面的提问格
assistant.answer(question, "gpt")




In [ ]:
# ========== 路径 B：再调一次 answer，provider 写成 "ollama" ==========

# 复用同一个 assistant；传入字符串 "ollama"（原笔记本如此；是否命中类内分支取决于 _get_client_and_model 的匹配规则）
# 理念：同一 question、换 provider 字符串，观察与上一格 GPT 路径的差异
assistant.answer(question, "ollama")
